<a href="https://colab.research.google.com/github/jceltruda/Projects-in-AI-and-ML/blob/main/Project_6/Task_3/02_model_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/SimRec')

Mounted at /content/drive


# SimRec Model Architecture

This notebook defines the SimRec model:
- Lambda schedulers
- Similarity distribution computation
- Data partitioning and sampling
- Evaluation functions (train / valid / test)
- The SimRec model

## 1. Setup

In [ ]:
!pip install -q torch numpy tqdm

In [ ]:
import sys
import copy
import torch
import random
import numpy as np
from collections import defaultdict
from multiprocessing import Process, Queue

## 2. Lambda Schedulers

Control the weight of the similarity distillation loss during training.

In [ ]:
class LinearScheduleWithWarmup:
    """Lambda scheduler: linear warmup then linear decay."""
    def __init__(self, lambd, warmup_steps, lamb_steps):
        self.lambd = 0
        self.warmup_steps = warmup_steps
        self.lamb_steps = lamb_steps
        self.warmup_alpha = lambd / warmup_steps
        self.alpha = lambd / (warmup_steps - lamb_steps)
        self.bias = lambd * (1 - (warmup_steps / (warmup_steps - lamb_steps)))
        self.current_step = -1
        self.step()

    def get_lambd(self):
        return max(self.lambd, 0)

    def step(self):
        self.current_step += 1
        if self.current_step < self.warmup_steps:
            self.lambd = self.warmup_alpha * self.current_step
        else:
            self.lambd = self.alpha * self.current_step + self.bias


class NoneSchedule:
    """Constant lambda (no scheduling)."""
    def __init__(self, lambd):
        self.lambd = lambd

    def get_lambd(self):
        return self.lambd

    def step(self):
        pass

## 3. Similarity Distribution

In [ ]:
PAD_IDX = 0

def create_similarity_distirbution(similarity_indices, similarity_values, temperature, positive_indices):
    """Build a soft target distribution from precomputed similarity scores."""
    num_items = similarity_indices.shape[0]
    num_positives = positive_indices.shape[0]

    # (num_positives, top_k_similar)
    pos_similarity_indices = torch.index_select(similarity_indices, index=positive_indices, dim=0)
    pos_similarity_values = torch.index_select(similarity_values, index=positive_indices, dim=0)

    # (num_positives, num_items)
    similarities = torch.full((num_positives, num_items), fill_value=-float('inf'), device=similarity_indices.device)
    similarities.scatter_(dim=1, index=pos_similarity_indices, src=pos_similarity_values)

    similarities /= temperature
    distribution = torch.nn.functional.softmax(similarities, dim=-1)
    return distribution

## 4. Data Utilities

In [ ]:
def random_neq(l, r, s):
    """Sample a random integer in [l, r) that is NOT in set s."""
    t = np.random.randint(l, r)
    while t in s:
        t = np.random.randint(l, r)
    return t


def sample_function(user_train, usernum, itemnum, batch_size, maxlen, result_queue, SEED):
    """Background worker that continuously generates training batches."""
    def sample():
        user = np.random.randint(1, usernum + 1)
        while len(user_train[user]) <= 1:
            user = np.random.randint(1, usernum + 1)

        seq = np.zeros([maxlen], dtype=np.int32)
        pos = np.zeros([maxlen], dtype=np.int32)
        neg = np.zeros([maxlen], dtype=np.int32)
        nxt = user_train[user][-1]
        idx = maxlen - 1

        ts = set(user_train[user])
        for i in reversed(user_train[user][:-1]):
            seq[idx] = i
            pos[idx] = nxt
            if nxt != 0:
                neg[idx] = random_neq(1, itemnum + 1, ts)
            nxt = i
            idx -= 1
            if idx == -1:
                break
        return user, seq, pos, neg

    np.random.seed(SEED)
    while True:
        one_batch = []
        for i in range(batch_size):
            one_batch.append(sample())
        result_queue.put(zip(*one_batch))


class WarpSampler(object):
    """Multi-process batch sampler for training data."""
    def __init__(self, User, usernum, itemnum, batch_size=64, maxlen=10, n_workers=1):
        self.result_queue = Queue(maxsize=n_workers * 10)
        self.processors = []
        for i in range(n_workers):
            self.processors.append(
                Process(target=sample_function, args=(User,
                                                      usernum,
                                                      itemnum,
                                                      batch_size,
                                                      maxlen,
                                                      self.result_queue,
                                                      np.random.randint(2e9)
                                                      )))
            self.processors[-1].daemon = True
            self.processors[-1].start()

    def next_batch(self):
        return self.result_queue.get()

    def close(self):
        for p in self.processors:
            p.terminate()
            p.join()

In [ ]:
def data_partition(fname, augmentations_fname=None):
    """Split user interaction sequences into train / valid / test.

    For each user with >= 3 interactions:
      - train = all but last 2
      - valid = second-to-last
      - test  = last
    """
    usernum = 0
    itemnum = 0
    User = defaultdict(list)
    user_train = {}
    user_valid = {}
    user_test = {}

    with open(fname, 'r') as f:
        for line in f:
            u, i = line.rstrip().split(' ')
            u = int(u)
            i = int(i)
            usernum = max(u, usernum)
            itemnum = max(i, itemnum)
            User[u].append(i)

    if augmentations_fname is not None:
        with open(f'data/{augmentations_fname}.txt', 'r') as f:
            for line in f:
                u, i = line.rstrip().split(' ')
                u = int(u)
                i = int(i)
                usernum = max(u, usernum)
                itemnum = max(i, itemnum)
                User[u].append(i)

    for user in User:
        nfeedback = len(User[user])
        if nfeedback < 3:
            user_train[user] = User[user]
            user_valid[user] = []
            user_test[user] = []
        else:
            user_train[user] = User[user][:-2]
            user_valid[user] = []
            user_valid[user].append(User[user][-2])
            user_test[user] = []
            user_test[user].append(User[user][-1])

    return [user_train, user_valid, user_test, usernum, itemnum]

## 5. Evaluation Functions

In [ ]:
def evaluate_test(model, dataset, args):
    """Evaluate on the test set using NDCG@10 and HR@10."""
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)

    NDCG = 0.0
    HT = 0.0
    valid_user = 0.0
    id_hr = defaultdict(list)
    id_ndcg = defaultdict(list)

    if usernum > 10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = range(1, usernum + 1)

    for u in users:
        if len(train[u]) < 1 or len(test[u]) < 1:
            continue

        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        seq[idx] = valid[u][0]
        idx -= 1
        for i in reversed(train[u]):
            seq[idx] = i
            idx -= 1
            if idx == -1:
                break

        rated = set(train[u])
        rated.add(0)
        item_idx = [test[u][0]]
        for _ in range(100):
            t = np.random.randint(1, itemnum + 1)
            while t in rated:
                t = np.random.randint(1, itemnum + 1)
            item_idx.append(t)

        predictions = -model.predict(*[np.array(l) for l in [[u], [seq], item_idx]])
        predictions = predictions[0]

        rank = predictions.argsort().argsort()[0].item()

        valid_user += 1
        if rank < 10:
            ndcg = 1 / np.log2(rank + 2)
            NDCG += ndcg
            HT += 1
            id_hr[item_idx[0]].append(1)
            id_ndcg[item_idx[0]].append(ndcg)
        else:
            id_hr[item_idx[0]].append(0)
            id_ndcg[item_idx[0]].append(0)

        if valid_user % 100 == 0:
            print('.', end="")
            sys.stdout.flush()

    return (NDCG / valid_user, HT / valid_user), id_hr, id_ndcg


def evaluate_valid(model, dataset, args):
    """Evaluate on the validation set using NDCG@10 and HR@10."""
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)

    NDCG = 0.0
    HT = 0.0
    valid_user = 0.0
    id_hr = defaultdict(list)
    id_ndcg = defaultdict(list)

    if usernum > 10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = range(1, usernum + 1)

    for u in users:
        if len(train[u]) < 1 or len(valid[u]) < 1:
            continue

        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        for i in reversed(train[u]):
            seq[idx] = i
            idx -= 1
            if idx == -1:
                break

        rated = set(train[u])
        rated.add(0)
        item_idx = [valid[u][0]]
        for _ in range(100):
            t = np.random.randint(1, itemnum + 1)
            while t in rated:
                t = np.random.randint(1, itemnum + 1)
            item_idx.append(t)

        predictions = -model.predict(*[np.array(l) for l in [[u], [seq], item_idx]])
        predictions = predictions[0]

        rank = predictions.argsort().argsort()[0].item()

        valid_user += 1
        if rank < 10:
            ndcg = 1 / np.log2(rank + 2)
            NDCG += ndcg
            HT += 1
            id_hr[item_idx[0]].append(1)
            id_ndcg[item_idx[0]].append(ndcg)
        else:
            id_hr[item_idx[0]].append(0)
            id_ndcg[item_idx[0]].append(0)

        if valid_user % 100 == 0:
            print('.', end="")
            sys.stdout.flush()

    return (NDCG / valid_user, HT / valid_user), id_hr, id_ndcg


def evaluate_train(model, dataset, args):
    """Evaluate on the training set using NDCG@10 and HR@10."""
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)

    NDCG = 0.0
    HT = 0.0
    valid_user = 0.0
    id_hr = defaultdict(list)
    id_ndcg = defaultdict(list)

    if usernum > 10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = range(1, usernum + 1)

    for u in users:
        if len(train[u]) < 1:
            continue

        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        for i in reversed(train[u][:-1]):
            seq[idx] = i
            idx -= 1
            if idx == -1:
                break

        rated = set(train[u][:-1])
        rated.add(0)
        item_idx = [train[u][-1]]
        for _ in range(100):
            t = np.random.randint(1, itemnum + 1)
            while t in rated:
                t = np.random.randint(1, itemnum + 1)
            item_idx.append(t)

        predictions = -model.predict(*[np.array(l) for l in [[u], [seq], item_idx]])
        predictions = predictions[0]

        rank = predictions.argsort().argsort()[0].item()

        valid_user += 1
        if rank < 10:
            ndcg = 1 / np.log2(rank + 2)
            NDCG += ndcg
            HT += 1
            id_hr[item_idx[0]].append(1)
            id_ndcg[item_idx[0]].append(ndcg)
        else:
            id_hr[item_idx[0]].append(0)
            id_ndcg[item_idx[0]].append(0)

        if valid_user % 100 == 0:
            print('.', end="")
            sys.stdout.flush()

    return (NDCG / valid_user, HT / valid_user), id_hr, id_ndcg

## 6. SimRec Model Definition

In [ ]:
class PointWiseFeedForward(torch.nn.Module):
    """Two-layer 1D-convolution feed-forward block with residual connection."""
    def __init__(self, hidden_units, dropout_rate):
        super(PointWiseFeedForward, self).__init__()
        self.conv1 = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout1 = torch.nn.Dropout(p=dropout_rate)
        self.relu = torch.nn.ReLU()
        self.conv2 = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout2 = torch.nn.Dropout(p=dropout_rate)

    def forward(self, inputs):
        outputs = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(inputs.transpose(-1, -2))))))
        outputs = outputs.transpose(-1, -2)  # Conv1D requires (N, C, Length)
        outputs += inputs
        return outputs


class SimRec(torch.nn.Module):
    """Sequential Recommendation with Similarity Integration.

    Transformer-based architecture (based on SASRec) extended with
    item-similarity distillation to mitigate the cold-start problem.
    """
    def __init__(self, user_num, item_num, args):
        super(SimRec, self).__init__()
        self.user_num = user_num
        self.item_num = item_num
        self.dev = args.device

        self.item_emb = torch.nn.Embedding(self.item_num + 1, args.hidden_units, padding_idx=0)
        self.pos_emb = torch.nn.Embedding(args.maxlen, args.hidden_units)
        self.emb_dropout = torch.nn.Dropout(p=args.dropout_rate)

        self.attention_layernorms = torch.nn.ModuleList()
        self.attention_layers = torch.nn.ModuleList()
        self.forward_layernorms = torch.nn.ModuleList()
        self.forward_layers = torch.nn.ModuleList()

        self.last_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)

        for _ in range(args.num_blocks):
            new_attn_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
            self.attention_layernorms.append(new_attn_layernorm)

            new_attn_layer = torch.nn.MultiheadAttention(args.hidden_units,
                                                         args.num_heads,
                                                         args.dropout_rate)
            self.attention_layers.append(new_attn_layer)

            new_fwd_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
            self.forward_layernorms.append(new_fwd_layernorm)

            new_fwd_layer = PointWiseFeedForward(args.hidden_units, args.dropout_rate)
            self.forward_layers.append(new_fwd_layer)

        self.reset_parameters()

    def reset_parameters(self):
        for name, param in self.named_parameters():
            try:
                torch.nn.init.xavier_normal_(param.data)
            except:
                pass  # ignore layers that fail init

    def log2feats(self, log_seqs):
        """Encode a sequence of item IDs into contextual representations."""
        if not torch.is_tensor(log_seqs):
            log_seqs = torch.LongTensor(log_seqs).to(self.dev)
        seqs = self.item_emb(log_seqs)
        seqs *= self.item_emb.embedding_dim ** 0.5
        positions = np.tile(np.array(range(log_seqs.shape[1])), [log_seqs.shape[0], 1])
        seqs += self.pos_emb(torch.LongTensor(positions).to(self.dev))
        seqs = self.emb_dropout(seqs)

        timeline_mask = log_seqs == 0
        seqs *= ~timeline_mask.unsqueeze(-1)

        tl = seqs.shape[1]
        attention_mask = ~torch.tril(torch.ones((tl, tl), dtype=torch.bool, device=self.dev))

        for i in range(len(self.attention_layers)):
            seqs = torch.transpose(seqs, 0, 1)
            Q = self.attention_layernorms[i](seqs)
            mha_outputs, _ = self.attention_layers[i](Q, seqs, seqs,
                                                      attn_mask=attention_mask)
            seqs = Q + mha_outputs
            seqs = torch.transpose(seqs, 0, 1)

            seqs = self.forward_layernorms[i](seqs)
            seqs = self.forward_layers[i](seqs)
            seqs *= ~timeline_mask.unsqueeze(-1)

        log_feats = self.last_layernorm(seqs)
        return log_feats

    def forward(self, user_ids, log_seqs, pos_seqs, neg_seqs):
        """Training forward pass. Returns pos_logits, neg_logits, and full logits."""
        log_feats = self.log2feats(log_seqs)

        if not torch.is_tensor(pos_seqs):
            pos_seqs = torch.LongTensor(pos_seqs).to(self.dev)
        if not torch.is_tensor(neg_seqs):
            neg_seqs = torch.LongTensor(neg_seqs).to(self.dev)

        pos_embs = self.item_emb(pos_seqs)
        neg_embs = self.item_emb(neg_seqs)

        pos_logits = (log_feats * pos_embs).sum(dim=-1)
        neg_logits = (log_feats * neg_embs).sum(dim=-1)

        embed = self.item_emb.weight
        logits = log_feats @ embed.T
        return pos_logits, neg_logits, logits

    def predict(self, user_ids, log_seqs, item_indices):
        """Inference: score candidate items for the last position."""
        log_feats = self.log2feats(log_seqs)
        final_feat = log_feats[:, -1, :]

        if not torch.is_tensor(item_indices):
            item_indices = torch.LongTensor(item_indices).to(self.dev)

        item_embs = self.item_emb(item_indices)
        logits = item_embs.matmul(final_feat.unsqueeze(-1)).squeeze(-1)
        return logits

## 7. Export Module

In [ ]:
%%writefile simrec_module.py
"""
SimRec: Sequential Recommendation with Similarity Integration
"""
import sys
import copy
import torch
import random
import numpy as np
from collections import defaultdict
from multiprocessing import Process, Queue

PAD_IDX = 0

# Lambda Schedulers
class LinearScheduleWithWarmup:
    def __init__(self, lambd, warmup_steps, lamb_steps):
        self.lambd = 0
        self.warmup_steps = warmup_steps
        self.lamb_steps = lamb_steps
        self.warmup_alpha = lambd / warmup_steps
        self.alpha = lambd / (warmup_steps - lamb_steps)
        self.bias = lambd * (1 - (warmup_steps / (warmup_steps - lamb_steps)))
        self.current_step = -1
        self.step()

    def get_lambd(self):
        return max(self.lambd, 0)

    def step(self):
        self.current_step += 1
        if self.current_step < self.warmup_steps:
            self.lambd = self.warmup_alpha * self.current_step
        else:
            self.lambd = self.alpha * self.current_step + self.bias


class NoneSchedule:
    def __init__(self, lambd):
        self.lambd = lambd

    def get_lambd(self):
        return self.lambd

    def step(self):
        pass

# Similarity Distribution
def create_similarity_distirbution(similarity_indices, similarity_values, temperature, positive_indices):
    num_items = similarity_indices.shape[0]
    num_positives = positive_indices.shape[0]
    pos_similarity_indices = torch.index_select(similarity_indices, index=positive_indices, dim=0)
    pos_similarity_values = torch.index_select(similarity_values, index=positive_indices, dim=0)
    similarities = torch.full((num_positives, num_items), fill_value=-float('inf'), device=similarity_indices.device)
    similarities.scatter_(dim=1, index=pos_similarity_indices, src=pos_similarity_values)
    similarities /= temperature
    distribution = torch.nn.functional.softmax(similarities, dim=-1)
    return distribution

# Data Utilities
def random_neq(l, r, s):
    t = np.random.randint(l, r)
    while t in s:
        t = np.random.randint(l, r)
    return t


def sample_function(user_train, usernum, itemnum, batch_size, maxlen, result_queue, SEED):
    def sample():
        user = np.random.randint(1, usernum + 1)
        while len(user_train[user]) <= 1:
            user = np.random.randint(1, usernum + 1)
        seq = np.zeros([maxlen], dtype=np.int32)
        pos = np.zeros([maxlen], dtype=np.int32)
        neg = np.zeros([maxlen], dtype=np.int32)
        nxt = user_train[user][-1]
        idx = maxlen - 1
        ts = set(user_train[user])
        for i in reversed(user_train[user][:-1]):
            seq[idx] = i
            pos[idx] = nxt
            if nxt != 0:
                neg[idx] = random_neq(1, itemnum + 1, ts)
            nxt = i
            idx -= 1
            if idx == -1:
                break
        return user, seq, pos, neg

    np.random.seed(SEED)
    while True:
        one_batch = []
        for i in range(batch_size):
            one_batch.append(sample())
        result_queue.put(zip(*one_batch))


class WarpSampler(object):
    def __init__(self, User, usernum, itemnum, batch_size=64, maxlen=10, n_workers=1):
        self.result_queue = Queue(maxsize=n_workers * 10)
        self.processors = []
        for i in range(n_workers):
            self.processors.append(
                Process(target=sample_function, args=(User, usernum, itemnum,
                                                      batch_size, maxlen,
                                                      self.result_queue,
                                                      np.random.randint(2e9))))
            self.processors[-1].daemon = True
            self.processors[-1].start()

    def next_batch(self):
        return self.result_queue.get()

    def close(self):
        for p in self.processors:
            p.terminate()
            p.join()


def data_partition(fname, augmentations_fname=None):
    usernum = 0
    itemnum = 0
    User = defaultdict(list)
    user_train = {}
    user_valid = {}
    user_test = {}
    with open(fname, 'r') as f:
        for line in f:
            u, i = line.rstrip().split(' ')
            u = int(u)
            i = int(i)
            usernum = max(u, usernum)
            itemnum = max(i, itemnum)
            User[u].append(i)
    if augmentations_fname is not None:
        with open(f'data/{augmentations_fname}.txt', 'r') as f:
            for line in f:
                u, i = line.rstrip().split(' ')
                u = int(u)
                i = int(i)
                usernum = max(u, usernum)
                itemnum = max(i, itemnum)
                User[u].append(i)
    for user in User:
        nfeedback = len(User[user])
        if nfeedback < 3:
            user_train[user] = User[user]
            user_valid[user] = []
            user_test[user] = []
        else:
            user_train[user] = User[user][:-2]
            user_valid[user] = [User[user][-2]]
            user_test[user] = [User[user][-1]]
    return [user_train, user_valid, user_test, usernum, itemnum]

# Evaluation Functions
def evaluate_test(model, dataset, args):
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)
    NDCG = 0.0; HT = 0.0; valid_user = 0.0
    id_hr = defaultdict(list); id_ndcg = defaultdict(list)
    if usernum > 10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = range(1, usernum + 1)
    for u in users:
        if len(train[u]) < 1 or len(test[u]) < 1: continue
        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        seq[idx] = valid[u][0]; idx -= 1
        for i in reversed(train[u]):
            seq[idx] = i; idx -= 1
            if idx == -1: break
        rated = set(train[u]); rated.add(0)
        item_idx = [test[u][0]]
        for _ in range(100):
            t = np.random.randint(1, itemnum + 1)
            while t in rated: t = np.random.randint(1, itemnum + 1)
            item_idx.append(t)
        predictions = -model.predict(*[np.array(l) for l in [[u], [seq], item_idx]])
        predictions = predictions[0]
        rank = predictions.argsort().argsort()[0].item()
        valid_user += 1
        if rank < 10:
            ndcg = 1 / np.log2(rank + 2); NDCG += ndcg; HT += 1
            id_hr[item_idx[0]].append(1); id_ndcg[item_idx[0]].append(ndcg)
        else:
            id_hr[item_idx[0]].append(0); id_ndcg[item_idx[0]].append(0)
        if valid_user % 100 == 0: print('.', end=""); sys.stdout.flush()
    return (NDCG / valid_user, HT / valid_user), id_hr, id_ndcg


def evaluate_valid(model, dataset, args):
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)
    NDCG = 0.0; HT = 0.0; valid_user = 0.0
    id_hr = defaultdict(list); id_ndcg = defaultdict(list)
    if usernum > 10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = range(1, usernum + 1)
    for u in users:
        if len(train[u]) < 1 or len(valid[u]) < 1: continue
        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        for i in reversed(train[u]):
            seq[idx] = i; idx -= 1
            if idx == -1: break
        rated = set(train[u]); rated.add(0)
        item_idx = [valid[u][0]]
        for _ in range(100):
            t = np.random.randint(1, itemnum + 1)
            while t in rated: t = np.random.randint(1, itemnum + 1)
            item_idx.append(t)
        predictions = -model.predict(*[np.array(l) for l in [[u], [seq], item_idx]])
        predictions = predictions[0]
        rank = predictions.argsort().argsort()[0].item()
        valid_user += 1
        if rank < 10:
            ndcg = 1 / np.log2(rank + 2); NDCG += ndcg; HT += 1
            id_hr[item_idx[0]].append(1); id_ndcg[item_idx[0]].append(ndcg)
        else:
            id_hr[item_idx[0]].append(0); id_ndcg[item_idx[0]].append(0)
        if valid_user % 100 == 0: print('.', end=""); sys.stdout.flush()
    return (NDCG / valid_user, HT / valid_user), id_hr, id_ndcg


def evaluate_train(model, dataset, args):
    [train, valid, test, usernum, itemnum] = copy.deepcopy(dataset)
    NDCG = 0.0; HT = 0.0; valid_user = 0.0
    id_hr = defaultdict(list); id_ndcg = defaultdict(list)
    if usernum > 10000:
        users = random.sample(range(1, usernum + 1), 10000)
    else:
        users = range(1, usernum + 1)
    for u in users:
        if len(train[u]) < 1: continue
        seq = np.zeros([args.maxlen], dtype=np.int32)
        idx = args.maxlen - 1
        for i in reversed(train[u][:-1]):
            seq[idx] = i; idx -= 1
            if idx == -1: break
        rated = set(train[u][:-1]); rated.add(0)
        item_idx = [train[u][-1]]
        for _ in range(100):
            t = np.random.randint(1, itemnum + 1)
            while t in rated: t = np.random.randint(1, itemnum + 1)
            item_idx.append(t)
        predictions = -model.predict(*[np.array(l) for l in [[u], [seq], item_idx]])
        predictions = predictions[0]
        rank = predictions.argsort().argsort()[0].item()
        valid_user += 1
        if rank < 10:
            ndcg = 1 / np.log2(rank + 2); NDCG += ndcg; HT += 1
            id_hr[item_idx[0]].append(1); id_ndcg[item_idx[0]].append(ndcg)
        else:
            id_hr[item_idx[0]].append(0); id_ndcg[item_idx[0]].append(0)
        if valid_user % 100 == 0: print('.', end=""); sys.stdout.flush()
    return (NDCG / valid_user, HT / valid_user), id_hr, id_ndcg

# Model
class PointWiseFeedForward(torch.nn.Module):
    def __init__(self, hidden_units, dropout_rate):
        super(PointWiseFeedForward, self).__init__()
        self.conv1 = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout1 = torch.nn.Dropout(p=dropout_rate)
        self.relu = torch.nn.ReLU()
        self.conv2 = torch.nn.Conv1d(hidden_units, hidden_units, kernel_size=1)
        self.dropout2 = torch.nn.Dropout(p=dropout_rate)

    def forward(self, inputs):
        outputs = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(inputs.transpose(-1, -2))))))
        outputs = outputs.transpose(-1, -2)
        outputs += inputs
        return outputs


class SimRec(torch.nn.Module):
    def __init__(self, user_num, item_num, args):
        super(SimRec, self).__init__()
        self.user_num = user_num
        self.item_num = item_num
        self.dev = args.device
        self.item_emb = torch.nn.Embedding(item_num + 1, args.hidden_units, padding_idx=0)
        self.pos_emb = torch.nn.Embedding(args.maxlen, args.hidden_units)
        self.emb_dropout = torch.nn.Dropout(p=args.dropout_rate)
        self.attention_layernorms = torch.nn.ModuleList()
        self.attention_layers = torch.nn.ModuleList()
        self.forward_layernorms = torch.nn.ModuleList()
        self.forward_layers = torch.nn.ModuleList()
        self.last_layernorm = torch.nn.LayerNorm(args.hidden_units, eps=1e-8)
        for _ in range(args.num_blocks):
            self.attention_layernorms.append(torch.nn.LayerNorm(args.hidden_units, eps=1e-8))
            self.attention_layers.append(torch.nn.MultiheadAttention(args.hidden_units, args.num_heads, args.dropout_rate))
            self.forward_layernorms.append(torch.nn.LayerNorm(args.hidden_units, eps=1e-8))
            self.forward_layers.append(PointWiseFeedForward(args.hidden_units, args.dropout_rate))
        self.reset_parameters()

    def reset_parameters(self):
        for name, param in self.named_parameters():
            try: torch.nn.init.xavier_normal_(param.data)
            except: pass

    def log2feats(self, log_seqs):
        if not torch.is_tensor(log_seqs):
            log_seqs = torch.LongTensor(log_seqs).to(self.dev)
        seqs = self.item_emb(log_seqs)
        seqs *= self.item_emb.embedding_dim ** 0.5
        positions = np.tile(np.array(range(log_seqs.shape[1])), [log_seqs.shape[0], 1])
        seqs += self.pos_emb(torch.LongTensor(positions).to(self.dev))
        seqs = self.emb_dropout(seqs)
        timeline_mask = log_seqs == 0
        seqs *= ~timeline_mask.unsqueeze(-1)
        tl = seqs.shape[1]
        attention_mask = ~torch.tril(torch.ones((tl, tl), dtype=torch.bool, device=self.dev))
        for i in range(len(self.attention_layers)):
            seqs = torch.transpose(seqs, 0, 1)
            Q = self.attention_layernorms[i](seqs)
            mha_outputs, _ = self.attention_layers[i](Q, seqs, seqs, attn_mask=attention_mask)
            seqs = Q + mha_outputs
            seqs = torch.transpose(seqs, 0, 1)
            seqs = self.forward_layernorms[i](seqs)
            seqs = self.forward_layers[i](seqs)
            seqs *= ~timeline_mask.unsqueeze(-1)
        return self.last_layernorm(seqs)

    def forward(self, user_ids, log_seqs, pos_seqs, neg_seqs):
        log_feats = self.log2feats(log_seqs)
        if not torch.is_tensor(pos_seqs): pos_seqs = torch.LongTensor(pos_seqs).to(self.dev)
        if not torch.is_tensor(neg_seqs): neg_seqs = torch.LongTensor(neg_seqs).to(self.dev)
        pos_embs = self.item_emb(pos_seqs)
        neg_embs = self.item_emb(neg_seqs)
        pos_logits = (log_feats * pos_embs).sum(dim=-1)
        neg_logits = (log_feats * neg_embs).sum(dim=-1)
        logits = log_feats @ self.item_emb.weight.T
        return pos_logits, neg_logits, logits

    def predict(self, user_ids, log_seqs, item_indices):
        log_feats = self.log2feats(log_seqs)
        final_feat = log_feats[:, -1, :]
        if not torch.is_tensor(item_indices):
            item_indices = torch.LongTensor(item_indices).to(self.dev)
        item_embs = self.item_emb(item_indices)
        return item_embs.matmul(final_feat.unsqueeze(-1)).squeeze(-1)

Writing simrec_module.py
